# U4-03 · Limpieza, transformación e integración

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 4, apartados 4.4 a 4.8*

Ejecuta cada celda con **Mayús + Intro**. Todos los datos son ficticios y se crean dentro del propio notebook.

In [ ]:
import pandas as pd
import numpy as np
datos = pd.DataFrame({
    "fecha":    ["2026-03-02", "2026-03-02", "2026-03-03", "2026-03-03",
                 "2026-03-04", "2026-03-04", "2026-03-04", "2026-03-05"],
    "linea":    ["L1", "L2", "L1", "l2 ", "L1", "L2", "L2", "L1"],
    "piezas":   [410, 520, 385, 498, None, 505, 505, 402],
    "temp_c":   [21.5, 22.0, 250.0, 21.8, 22.4, 23.1, 23.1, None],
    "defectos": [6, 4, 9, 7, 5, None, None, 8],
})
datos

## 1. Limpieza

In [ ]:
d = datos.copy()
d["linea"] = d["linea"].str.strip().str.upper()    # "l2 " -> "L2"
d = d.drop_duplicates()                            # 8 -> 7 filas
fuera = (d["temp_c"] < -10) | (d["temp_c"] > 60)   # rango físico posible
d.loc[fuera, "temp_c"] = np.nan                    # 250 -> ausente
print(len(d), d.isna().sum().to_dict())
d

## 2. La imputación por la media reduce la varianza (ejemplo 4.3)

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(50, 5, 1000)
falta = rng.random(1000) < 0.3
s = pd.Series(np.where(falta, np.nan, x))
print("Ausentes:", falta.sum())
print("Varianza original:  ", round(x.var(ddof=1), 2))
print("Varianza observados:", round(s.var(), 2))
print("Varianza imputada:  ", round(s.fillna(s.mean()).var(), 2))

## 3. Escalado ante un atípico (ejemplo 4.4)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
t = np.array([41.2, 39.8, 40.5, 42.1, 40.9, 39.5, 41.7, 40.3, 51.0, 40.1]).reshape(-1, 1)
tabla = pd.DataFrame({"tiempo": t.ravel()})
for esc in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    tabla[type(esc).__name__] = esc.fit_transform(t).ravel().round(2)
tabla

In [ ]:
from scipy import stats
y = np.random.default_rng(1).lognormal(mean=2, sigma=0.6, size=500)
yt, lam = stats.boxcox(y)
print("Asimetría antes:", round(stats.skew(y), 2), "· después:", round(stats.skew(yt), 2), "· lambda:", round(lam, 3))

## 4. Codificación de categorías

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
lin = pd.DataFrame({"linea": ["L1", "L2", "L3", "L1"]})
oh = OneHotEncoder(sparse_output=False).fit(lin)
print(oh.get_feature_names_out())
print(oh.transform(lin))
grav = pd.DataFrame({"gravedad": ["leve", "grave", "crítica", "leve"]})
print(OrdinalEncoder(categories=[["leve", "grave", "crítica"]]).fit_transform(grav).ravel())

## 5. Ingeniería de características: del minuto a la hora (ejemplo 4.5)

In [ ]:
rng = np.random.default_rng(3)
idx = pd.date_range("2026-03-02 08:00", periods=180, freq="min")
sensor = pd.DataFrame({"temp_c": (60 + np.linspace(0, 6, 180) + rng.normal(0, 0.8, 180)).round(1)}, index=idx)

def pendiente(s):
    return np.polyfit(np.arange(len(s)), s.values, 1)[0]

por_hora = sensor["temp_c"].resample("h").agg(["mean", "max", pendiente])
por_hora.columns = ["media", "maximo", "pendiente"]
por_hora.round(3)

In [ ]:
import matplotlib.pyplot as plt
ax = sensor["temp_c"].plot(figsize=(7, 3), color="#1C8C96", lw=0.8, label="cada minuto")
por_hora["media"].plot(ax=ax, drawstyle="steps-post", color="#1F3A40", lw=2, label="media horaria")
ax.set_ylabel("Temperatura (ºC)"); ax.legend(); plt.show()

## 6. Integración de fuentes (ejemplo 4.6)

In [ ]:
erp = pd.DataFrame({"id_maquina": ["M-01", "M-02", "M-03"], "linea": ["L1", "L2", "L3"],
                    "horas_uso": [1520, 980, 2210]})
mant = pd.DataFrame({"maquina": ["m01", "m02", "m02", "m04"],
                     "fecha": ["2026-01-10", "2026-02-03", "2026-02-20", "2026-02-25"]})
mant["id_maquina"] = "M-" + mant["maquina"].str[1:]
averias = mant.groupby("id_maquina").size().rename("n_averias").reset_index()
erp.merge(averias, on="id_maquina", how="outer", indicator=True)

## 7. Todo en un objeto: ColumnTransformer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

numericas, categoricas = ["piezas", "temp_c"], ["linea"]
preparacion = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), categoricas),
])
X = preparacion.fit_transform(d[numericas + categoricas])
pd.DataFrame(X, columns=preparacion.get_feature_names_out()).round(2)

In [ ]:
# Un dato nuevo, con una línea que no existía y un ausente
nuevo = pd.DataFrame({"piezas": [None], "temp_c": [24.0], "linea": ["L9"]})
preparacion.transform(nuevo).round(2)

**Ejercicio.** Añade la columna `defectos` a las numéricas y comprueba cómo cambia la forma de `X`.

> Little, R. J. A. y Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3.ª ed.). Wiley. https://doi.org/10.1002/9781119482260